# レッスン 01 - AIエージェント入門

<strong>初心者向けAIエージェント</strong>コースの最初のレッスンへようこそ！

<strong>AIエージェント</strong>とは、大規模言語モデル（LLM）を推論エンジンとして使用し、ユーザーのために目標を達成するために、APIの呼び出し、データベースの照会、コードの実行などの<em>アクション</em>を現実世界で実行できるプログラムのことです。

このノートブックでは、最初のエージェントである、旅行先を推薦する<strong>トラベルエージェント</strong>を作成します。途中で次のことを学びます：

1. <strong>Microsoft Agent Framework</strong>を使ってMicrosoft Foundry Agent Serviceに接続する方法。
2. エージェントに<strong>ツール</strong>（呼び出せるシンプルなPython関数）を与える方法。
3. エージェントを実行し、その応答を検査する方法。
4. エージェントの応答をトークンごとにストリーミングする方法。


## セットアップ

このノートブックを実行する前に、次のことを確認してください:

1. **チャットモデルがデプロイされた Microsoft Foundry プロジェクト**（例: `gpt-5-mini`）。
2. **Azure CLI にログインしていること** — ターミナルで `az login` を実行してください。
3. **必要な環境変数を設定していること:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry プロジェクトのエンドポイント。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — デプロイされたモデルの名前。

以下のセルは必要な Python パッケージをインストールします。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## あなたの最初のエージェントを作成する

エージェントには二つのものが必要です：

- <strong>指示</strong> — それが「誰であるか」と「どのように振る舞うか」を伝えるもの（システムプロンプト）。
- <strong>ツール</strong> — エージェントが情報を取得したり行動を実行したりするために呼び出せる、`@tool` で装飾されたPython関数。

以下では、人気の休暇先のリストを返すシンプルなツールを定義しています。エージェントはユーザーが旅行のおすすめを求めたときにこのツールを使用します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ストリーミング応答

よりインタラクティブな体験のために、エージェントの応答を<strong>ストリーミング</strong>できます。完全な応答を待つ代わりに、エージェントは生成されるテキストのチャンクを逐次提供します。これは、チャットインターフェースでリアルタイムに出力を表示したい場合に特に便利です。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## まとめ

このレッスンでは次のことを学びました：

- `FoundryChatClient` を通じて Microsoft Foundry Agent Service に接続する <strong>プロバイダーを作成する</strong> 方法。
- エージェントがあなたの Python 関数を呼び出せるように、`@tool` デコレーターを使って <strong>ツールを定義する</strong> 方法。
- ユーザーメッセージで <strong>エージェントを実行し</strong>、その応答を表示する方法。
- リアルタイム出力のために <strong>レスポンスをストリーミングする</strong> 方法。

次のレッスンでは、エージェントフレームワークをより深く探求し、エージェントにより強力なツールや複数段階の推論機能を持たせる方法を学びます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
